In [1]:
%pip install pyproj requests pandas IPython datetime display python-dotenv

  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
Using cached python_dotenv-1.2.1-py3-none-any.whl (21 kB)
Note: you may need to restart the kernel to use updated packages.


# 🌦️ API 연동 및 좌표 변환 실습

이 문서는 공공데이터포털의 기상청 API와 에어코리아 API를 연동하는 파이썬 코드를 정리한 내용입니다.

## 1. 기상청 초단기실황조회 API 호출

기상청의 단기예보 조회 서비스 중 '초단기실황조회'를 사용하여 특정 격자 좌표의 실시간 날씨 데이터를 가져옵니다.

### 1.1 기본 API 호출 및 데이터 프레임 변환

기상청 API는 데이터를 암호화된 코드(PTY, REH 등)로 반환합니다. 아래는 암호화된 코드 그대로 출력하는 코드입니다. 
한글로 매핑하여 출력하는 코드는 1.2에 있습니다.

In [2]:
import requests
import pandas as pd
from datetime import datetime
from IPython.display import display
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

# ── 인증키 / 엔드포인트 설정 ─────────────────────────────────────────────────
WEATHER_API_KEY = os.getenv("WEATHER_API_KEY")
WEATHER_API_URL = "http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtNcst"

# ── 현재 시각 기준 날짜/시간 자동 설정 ───────────────────────────────────────
now = datetime.now()
base_date = now.strftime("%Y%m%d")   # 조회 기준 날짜 (예: 20260225)
base_time = now.strftime("%H%M")     # 조회 기준 시간 (예: 1400)

# ── 격자 좌표 설정 (위경도 → 격자 변환은 아래 좌표변환 셀 참고) ──────────────
nx = 57   # 경기도 평택 인근 예시
ny = 74

# ── 요청 파라미터 구성 ────────────────────────────────────────────────────────
params = {
    "pageNo":     "1",
    "numOfRows":  "1000",    # 최대 조회 행 수
    "dataType":   "JSON",
    "base_date":  base_date,
    "base_time":  base_time,
    "nx":         nx,
    "ny":         ny,
    "serviceKey": WEATHER_API_KEY,
}

# ── API 호출 및 결과 처리 ────────────────────────────────────────────────────
try:
    response = requests.get(WEATHER_API_URL, params=params)

    if response.status_code == 200:
        data = response.json()
        result_code = data["response"]["header"]["resultCode"]  # "00" 이면 정상

        if result_code == "00":
            items = data["response"]["body"]["items"]["item"]   # 관측 항목 리스트

            # 관측 항목 리스트를 DataFrame으로 변환
            weather_df = pd.DataFrame(items)
            display(weather_df)
        else:
            print(f"API 오류: {result_code} - {data['response']['header']['resultMsg']}")

    else:
        # HTTP 레벨 오류 (네트워크/서버 문제)
        print(f"HTTP 오류: {response.status_code}")
        print(response.text)

except Exception as e:
    print(f"연동 중 오류가 발생했습니다: {e}")

,baseDate,baseTime,category,nx,ny,obsrValue
0,20260225,1200,PTY,57,74,0
1,20260225,1200,REH,57,74,66
2,20260225,1200,RN1,57,74,0
3,20260225,1200,T1H,57,74,10.3
4,20260225,1200,UUU,57,74,0.1
5,20260225,1200,VEC,57,74,191
6,20260225,1200,VVV,57,74,0.5
7,20260225,1200,WSD,57,74,0.5


### 1.2 영문/한글 매핑 포함

기상청 API는 데이터를 암호화된 코드(PTY, REH 등)로 반환하므로, 이를 알아보기 쉽게 한글로 매핑하여 출력합니다.

In [3]:
import requests
import pandas as pd
from datetime import datetime
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

# ── 인증키 / 엔드포인트 설정 ─────────────────────────────────────────────────
WEATHER_API_KEY = os.getenv("WEATHER_API_KEY")
WEATHER_API_URL = "http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtNcst"

# ── 현재 시각 기준 날짜/시간 자동 설정 ───────────────────────────────────────
now = datetime.now()
base_date = now.strftime("%Y%m%d")   # 조회 기준 날짜 (예: 20260225)
base_time = now.strftime("%H%M")     # 조회 기준 시간 (예: 1400)

# ── 격자 좌표 설정 (위경도 → 격자 변환은 아래 좌표변환 셀 참고) ──────────────
nx = 57   # 경기도 평택 인근 예시
ny = 74

# ── 요청 파라미터 구성 ────────────────────────────────────────────────────────
params = {
    "pageNo":     "1",
    "numOfRows":  "1000",    # 최대 조회 행 수
    "dataType":   "JSON",
    "base_date":  base_date,
    "base_time":  base_time,
    "nx":         nx,
    "ny":         ny,
    "serviceKey": WEATHER_API_KEY,
}

# ── 기상 코드 → 한글 설명 매핑 ───────────────────────────────────────────────
category_map = {
    "T1H": "기온(°C)",
    "RN1": "1시간 강수량(mm)",
    "UUU": "동서바람성분(m/s)",
    "VVV": "남북바람성분(m/s)",
    "REH": "습도(%)",
    "PTY": "강수형태",
    "VEC": "풍향(deg)",
    "WSD": "풍속(m/s)",
}

# ── 강수형태 코드 → 한글 매핑 ────────────────────────────────────────────────
pty_map = {
    "0": "없음", "1": "비", "2": "비/눈", "3": "눈",
    "5": "빗방울", "6": "진눈깨비", "7": "눈날림"
}

# ── API 호출 및 결과 처리 ─────────────────────────────────────────────────────
try:
    response = requests.get(WEATHER_API_URL, params=params)

    if response.status_code == 200:
        data = response.json()
        result_code = data["response"]["header"]["resultCode"]  # "00" 이면 정상

        if result_code == "00":
            items = data["response"]["body"]["items"]["item"]   # 관측 항목 리스트

            # 각 항목을 한글 컬럼명으로 변환하여 rows 리스트에 누적
            rows = []
            for item in items:
                cat   = item["category"]    # 기상 요소 코드 (T1H, RN1 등)
                value = item["obsrValue"]   # 관측값

                # 강수형태(PTY)는 숫자 코드를 한글로 변환
                if cat == "PTY":
                    value = pty_map.get(str(value), value)

                rows.append({
                    "항목코드":   cat,
                    "항목명":     category_map.get(cat, cat),   # 매핑 없으면 코드 그대로
                    "관측값":     value,
                    "기준날짜":   item["baseDate"],
                    "기준시간":   item["baseTime"],
                })

            # rows 리스트를 DataFrame으로 변환
            df = pd.DataFrame(rows)

            print(f"\n--- 초단기실황 조회 성공 ({base_date} {base_time} 기준) ---")
            print(f"격자좌표: nx={nx}, ny={ny}\n")
            print(df.to_string(index=False))
            print(df.describe())

        else:
            print(f"API 오류: {result_code} - {data['response']['header']['resultMsg']}")

    else:
        # HTTP 레벨 오류 (네트워크/서버 문제)
        print(f"HTTP 오류: {response.status_code}")
        print(response.text)

except Exception as e:
    print(f"연동 중 오류가 발생했습니다: {e}")


--- 초단기실황 조회 성공 (20260225 1241 기준) ---
격자좌표: nx=57, ny=74

항목코드         항목명  관측값     기준날짜 기준시간
 PTY        강수형태   없음 20260225 1200
 REH       습도(%)   66 20260225 1200
 RN1 1시간 강수량(mm)    0 20260225 1200
 T1H      기온(°C) 10.3 20260225 1200
 UUU 동서바람성분(m/s)  0.1 20260225 1200
 VEC     풍향(deg)  191 20260225 1200
 VVV 남북바람성분(m/s)  0.5 20260225 1200
 WSD     풍속(m/s)  0.5 20260225 1200
       항목코드   항목명  관측값      기준날짜  기준시간
count     8     8    8         8     8
unique    8     8    7         1     1
top     PTY  강수형태  0.5  20260225  1200
freq      1     1    2         8     8


실행결과 데이터프레임으로 출력하기

In [4]:
df

,항목코드,항목명,관측값,기준날짜,기준시간
0,PTY,강수형태,없음,20260225,1200
1,REH,습도(%),66,20260225,1200
2,RN1,1시간 강수량(mm),0,20260225,1200
3,T1H,기온(°C),10.3,20260225,1200
4,UUU,동서바람성분(m/s),0.1,20260225,1200
5,VEC,풍향(deg),191,20260225,1200
6,VVV,남북바람성분(m/s),0.5,20260225,1200
7,WSD,풍속(m/s),0.5,20260225,1200


---
## 2. 좌표 변환 로직 (위경도 ↔ 격자 ↔ TM)

기상청 API는 **격자 좌표(nx, ny)**를 사용하고, 에어코리아(측정소) API는 **TM 좌표(tmX, tmY)**를 사용합니다. 일반적인 **위도/경도**를 각 API에 맞는 좌표계로 변환하는 함수입니다.

### 2.1 위도/경도 → 격자(LCC) / TM 변환

In [5]:
import math
from pyproj import Transformer

# ── 기상청 LCC(Lambert Conformal Conic) 격자 변환 상수 ───────────────────────
RE     = 6371.00877   # 지구 반경 (km)
GRID   = 5.0          # 격자 간격 (km)
SLAT1  = 30.0         # 표준 위도 1 (°N)
SLAT2  = 60.0         # 표준 위도 2 (°N)
OLON   = 126.0        # 기준점 경도 (°E)
OLAT   = 38.0         # 기준점 위도 (°N)
XO     = 43           # 기준점 X 격자 좌표
YO     = 136          # 기준점 Y 격자 좌표


def latlon_to_grid(lat, lon):
    """위도·경도 → 기상청 격자 좌표 (nx, ny)  [WEATHER_API 사용]"""
    DEGRAD = math.pi / 180.0        # 도 → 라디안 변환 계수
    re  = RE / GRID                 # 격자 반경

    # 표준 위도·기준점을 라디안으로 변환
    slat1 = SLAT1 * DEGRAD
    slat2 = SLAT2 * DEGRAD
    olon  = OLON  * DEGRAD
    olat  = OLAT  * DEGRAD

    # LCC 투영 계수 sn, sf, ro 계산
    sn = math.tan(math.pi * 0.25 + slat2 * 0.5) / math.tan(math.pi * 0.25 + slat1 * 0.5)
    sn = math.log(math.cos(slat1) / math.cos(slat2)) / math.log(sn)
    sf = math.tan(math.pi * 0.25 + slat1 * 0.5)
    sf = (sf ** sn) * math.cos(slat1) / sn
    ro = math.tan(math.pi * 0.25 + olat * 0.5)
    ro = re * sf / (ro ** sn)   # 기준점의 반지름 벡터

    # 입력 위도에 대한 반지름 벡터 및 경도 차이(θ) 계산
    ra = math.tan(math.pi * 0.25 + lat * DEGRAD * 0.5)
    ra = re * sf / (ra ** sn)
    theta = lon * DEGRAD - olon
    # θ를 (-π, π] 범위로 정규화
    if theta > math.pi:  theta -= 2.0 * math.pi
    if theta < -math.pi: theta += 2.0 * math.pi
    theta *= sn

    # 격자 좌표 계산 (반올림 후 정수 변환)
    nx = int(ra * math.sin(theta) + XO + 0.5)
    ny = int(ro - ra * math.cos(theta) + YO + 0.5)
    return nx, ny


def latlon_to_tm(lat, lon):
    """위도·경도 → TM 좌표 (tmX, tmY)  [NEARBY_STATION_API 사용]
    EPSG:4326(WGS84) → EPSG:2097(한국 중부원점 TM) 변환
    """
    transformer = Transformer.from_crs("epsg:4326", "epsg:2097", always_xy=False)
    tmy, tmx = transformer.transform(lat, lon)   # transform 반환 순서: (tmy, tmx)
    return tmx, tmy


# ── 테스트 ────────────────────────────────────────────────────────────────────
lat, lon = 37.2965, 126.9851       # 예시 좌표 (경기도 화성)

nx, ny   = latlon_to_grid(lat, lon)
tmx, tmy = latlon_to_tm(lat, lon)

print(f"입력 좌표       lat={lat}, lon={lon}")
print(f"기상청 격자     nx={nx}, ny={ny}          ← WEATHER_API 사용")
print(f"TM 좌표         tmX={tmx:.3f}, tmY={tmy:.3f}  ← NEARBY_STATION_API 사용")

입력 좌표       lat=37.2965, lon=126.9851
기상청 격자     nx=60, ny=121          ← WEATHER_API 사용
TM 좌표         tmX=198865.042, tmY=421613.284  ← NEARBY_STATION_API 사용


### 2.2 TM → 위도/경도 → 격자 좌표 (역변환)

In [6]:
from pyproj import Transformer

# ※ 이 셀은 위 좌표변환 셀(latlon_to_grid)이 먼저 실행되어 있어야 합니다.

def tm_to_latlon(tmx, tmy):
    """TM 좌표 (tmX, tmY) → 위도·경도
    EPSG:2097(한국 중부원점 TM) → EPSG:4326(WGS84) 역변환
    """
    transformer = Transformer.from_crs("epsg:2097", "epsg:4326", always_xy=False)
    lat, lon = transformer.transform(tmy, tmx)   # transform 반환 순서: (lat, lon)
    return lat, lon


def tm_to_grid(tmx, tmy):
    """TM 좌표 (tmX, tmY) → 기상청 격자 좌표 (nx, ny)  [WEATHER_API 사용]
    TM → 위도/경도 → 격자 좌표 순으로 2단계 변환
    """
    lat, lon = tm_to_latlon(tmx, tmy)
    return latlon_to_grid(lat, lon)   # 위 좌표변환 셀의 함수 재사용


# ── 테스트 (이전 셀 출력값으로 역변환 검증) ──────────────────────────────────
tmx_test, tmy_test = 198865.042, 421613.284   # latlon_to_tm(37.2965, 126.9851) 결과

lat_conv, lon_conv = tm_to_latlon(tmx_test, tmy_test)   # TM → 위도/경도
nx_conv,  ny_conv  = tm_to_grid(tmx_test, tmy_test)     # TM → 격자 좌표

print(f"입력 TM 좌표    tmX={tmx_test}, tmY={tmy_test}")
print(f"변환 위도/경도  lat={lat_conv:.4f}, lon={lon_conv:.4f}")
print(f"기상청 격자     nx={nx_conv}, ny={ny_conv}          ← WEATHER_API 사용")

입력 TM 좌표    tmX=198865.042, tmY=421613.284
변환 위도/경도  lat=37.2965, lon=126.9851
기상청 격자     nx=60, ny=121          ← WEATHER_API 사용


---

## 3. 에어코리아 대기오염정보 API 호출

한국환경공단의 에어코리아 API를 사용하여 **가장 가까운 측정소**를 찾고, 해당 측정소의 **미세먼지 농도**를 조회합니다.

### 3.1 측정소 조회 및 미세먼지 데이터 확인

In [7]:
import requests
import pandas as pd
from pyproj import Transformer
from IPython.display import display
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

# ── 인증키 / 엔드포인트 설정 ─────────────────────────────────────────────────
MISE_API_KEY = os.getenv("MISE_API_KEY")
NEARBY_STATION_API_URL = "https://apis.data.go.kr/B552584/MsrstnInfoInqireSvc/getNearbyMsrstnList"
MISE_API_URL = "https://apis.data.go.kr/B552584/ArpltnInforInqireSvc/getMsrstnAcctoRltmMesureDnsty"

# ── 등급 코드 → 한글 매핑 ────────────────────────────────────────────────────
grade_map = {"1": "좋음", "2": "보통", "3": "나쁨", "4": "매우나쁨"}


def get_nearby_station(lat, lon):
    """위도·경도 → 가장 가까운 에어코리아 측정소 조회
    위도/경도를 TM 좌표로 변환 후 NEARBY_STATION_API 호출
    Returns: (시도명, 측정소명) 또는 오류 시 (None, None)
    """
    # 위도/경도를 TM 좌표로 변환 (이전 좌표변환 셀의 함수 사용)
    tmx, tmy = latlon_to_tm(lat, lon)

    params = {
        "serviceKey": MISE_API_KEY,
        "returnType": "json",
        "tmX": tmx,
        "tmY": tmy,
    }

    try:
        response = requests.get(NEARBY_STATION_API_URL, params=params)
        if response.status_code != 200 or not response.text.strip():
            print(f"HTTP 오류: {response.status_code}")
            return None, None

        stations = response.json().get("response", {}).get("body", {}).get("items", [])
        # 거리(tm) 기준 오름차순 정렬 → 가장 가까운 측정소 선택
        stations = sorted(stations, key=lambda s: s["tm"])
        target = stations[0]
        sido_name    = target["addr"][:2]         # 주소 앞 2자리 = 시도명
        station_name = target["stationName"]
        print(f"인접 측정소: {station_name} ({sido_name})")
        return sido_name, station_name

    except Exception as e:
        print(f"측정소 조회 오류: {e}")
        return None, None


def get_mise_info(lat=37.2965, lon=126.9851):
    """위도·경도 기준 미세먼지(PM10, PM2.5) 현황 조회
    측정소 조회 → 대기오염 정보 조회 → 최신 데이터 반환
    Returns: 결과 dict 또는 오류 시 None
    """
    # 인접 측정소 조회
    sido_name, station_name = get_nearby_station(lat, lon)
    if not station_name:
        return None

    params = {
        "serviceKey": MISE_API_KEY,
        "returnType": "json",
        "pageNo":     "1",
        "stationName": station_name,
        "dataTerm":   "DAILY",   # 당일 데이터 조회
        "ver":        "1.3",
    }

    try:
        response = requests.get(MISE_API_URL, params=params)
        if response.status_code != 200 or not response.text.strip():
            print(f"HTTP 오류: {response.status_code}")
            return None

        mise_data = response.json().get("response", {}).get("body", {}).get("items", [])
        if not mise_data:
            print("데이터 없음")
            return None

        # dataTime 기준 내림차순 정렬 → 가장 최신 측정값 선택
        target_data = sorted(mise_data, key=lambda d: d["dataTime"], reverse=True)[0]

        # API 버전에 따라 등급 키 이름이 다를 수 있음 (pm10Grade / pm10Grade1h)
        pm10_grade = target_data.get("pm10Grade") or target_data.get("pm10Grade1h", "-")
        pm25_grade = target_data.get("pm25Grade") or target_data.get("pm25Grade1h", "-")

        result = {
            "측정소":     station_name,
            "지역":       sido_name,
            "측정시간":   target_data.get("dataTime"),
            "PM10 농도":  target_data.get("pm10Value"),
            "PM10 등급":  grade_map.get(str(pm10_grade), "-"),
            "PM2.5 농도": target_data.get("pm25Value"),
            "PM2.5 등급": grade_map.get(str(pm25_grade), "-"),
        }
        return result

    except Exception as e:
        print(f"미세먼지 조회 오류: {e}")
        return None


# ── 실행 및 결과 출력 ─────────────────────────────────────────────────────────
try:
    mise_result = get_mise_info()
    if mise_result:
        # 단일 행 dict를 DataFrame으로 변환하여 표 형태로 출력
        mise_df = pd.DataFrame([mise_result])
        display(mise_df)
    else:
        print("결과를 가져오지 못했습니다.")

except Exception as e:
    print(f"연동 중 오류가 발생했습니다: {e}")

인접 측정소: 천천동 (경기)


,측정소,지역,측정시간,PM10 농도,PM10 등급,PM2.5 농도,PM2.5 등급
0,천천동,경기,2026-02-25 12:00,50,보통,20,보통
